In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("volume_schema", "default", "Schema do Volume")
dbutils.widgets.text("volume_name", "inputs", "Nome do Volume")

CATALOG = dbutils.widgets.get("catalog").strip()
VOLUME_SCHEMA = dbutils.widgets.get("volume_schema").strip()
VOLUME_NAME = dbutils.widgets.get("volume_name").strip()

BRONZE_DB = "bronze"
VOLUME_PATH = f"/Volumes/workspace/default/inputs"

# Mapeamento: arquivo de origem -> tabela Bronze
SOURCES = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

print(f"Origem : {VOLUME_PATH}")
print(f"Destino: {CATALOG}.{BRONZE_DB}")

Origem : /Volumes/workspace/default/inputs
Destino: workspace.bronze


In [0]:
#Cria o database Bronze se nao tiver
spark.sql(
    f"""
    CREATE DATABASE IF NOT EXISTS {CATALOG}.{BRONZE_DB}
    COMMENT 'Camada Bronze: dados brutos do catálogo de filmes TMDB/IMDb, sem transformação'
    """
)

DataFrame[]

In [0]:
for file_name, table_name in SOURCES.items():
    # Lê o CSV
    df = spark.read.csv(f"{VOLUME_PATH}/{file_name}", header=True)

    # Acrescenta o timestamp do momento da inserção na Bronze
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    # Grava em formato Delta, modo Append
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_DB}.{table_name}")

    print(f"{BRONZE_DB}.{table_name} gravada")

bronze.tb_movies_info gravada
bronze.tb_movies_financials gravada
bronze.tb_movies_metrics gravada
bronze.tb_credits_and_tags gravada
bronze.tb_movies_reviews gravada


In [0]:
df = spark.table(f"{CATALOG}.{BRONZE_DB}.tb_movies_info")
display(df.limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T20:13:06.910Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T20:13:06.910Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T20:13:06.910Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T20:13:06.910Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T20:13:06.910Z


In [0]:
import requests
from datetime import date, datetime, timedelta

# Parâmetros de data no formato MM-DD-AAAA.
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data fim (MM-DD-AAAA)")

hoje = date.today()
data_inicio_formatada = dbutils.widgets.get("data_inicio").strip() or (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
data_fim_formatada = dbutils.widgets.get("data_fim").strip() or hoje.strftime("%m-%d-%Y")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)


def mock_resposta_api(data_inicio: str, data_fim: str) -> dict:
    """Simula a resposta do endpoint: uma cotação por dia útil do período."""
    inicio = datetime.strptime(data_inicio, "%m-%d-%Y").date()
    fim = datetime.strptime(data_fim, "%m-%d-%Y").date()
    dias = [inicio + timedelta(days=i) for i in range((fim - inicio).days + 1)]
    return {
        "value": [
            {"cotacaoCompra": round(5.40 + 0.01 * i, 4), "dataHoraCotacao": f"{dia:%Y-%m-%d} 13:00:00.000"}
            for i, dia in enumerate(dias)
            if dia.weekday() < 5
        ]
    }


try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    resposta = response.json()
    print("Fonte: API do Banco Central")
except Exception as erro:
    resposta = mock_resposta_api(data_inicio_formatada, data_fim_formatada)
    print(f"Fonte: MOCK (API indisponível: {erro})")

cotacoes = resposta["value"]

print(f"Período: {data_inicio_formatada} a {data_fim_formatada}")
print(cotacoes)

Fonte: API do Banco Central
Período: 10-09-2026 a 10-16-2026
[{'cotacaoCompra': 4.9886, 'dataHoraCotacao': '2026-10-09 13:07:30.746597'}, {'cotacaoCompra': 5.009, 'dataHoraCotacao': '2026-10-13 13:04:26.866133'}, {'cotacaoCompra': 5.0435, 'dataHoraCotacao': '2026-10-14 13:05:17.833426'}, {'cotacaoCompra': 5.0515, 'dataHoraCotacao': '2026-10-15 13:05:33.305854'}, {'cotacaoCompra': 5.082, 'dataHoraCotacao': '2026-10-16 13:04:14.548156'}]


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])

df_bronze_cotacao = (
    spark.createDataFrame(
        [(c["dataHoraCotacao"], float(c["cotacaoCompra"])) for c in cotacoes],
        schema=schema,
    )
    .withColumn("ingestion_datetime", F.current_timestamp())
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

df_bronze_cotacao.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.tb_cotacao_dolar")

display(spark.table(f"{CATALOG}.bronze.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-10-09 13:07:30.746597,4.9886,2026-10-16T20:13:21.985Z
2026-10-13 13:04:26.866133,5.009,2026-10-16T20:13:21.985Z
2026-10-14 13:05:17.833426,5.0435,2026-10-16T20:13:21.985Z
2026-10-15 13:05:33.305854,5.0515,2026-10-16T20:13:21.985Z
2026-10-16 13:04:14.548156,5.082,2026-10-16T20:13:21.985Z
